# 멈추고, 보고, 승인하기 · breakpoint

강의 오후 ③ 가운데 (Academy 모듈 3 · L2 Breakpoints)

| 칸 | 보여 주는 것 |
| --- | --- |
| ① 멈춘다 | `interruptBefore: ["tools"]` 로 도구를 쓰기 전에 멈춘다. stream 이 `__interrupt__` 를 받고 끝난다 |
| ② 본다 | `getState` 의 `next` 로 어디서 멈췄는지, `values` 로 모델이 하려는 일을 본다 |
| ③ 이어 간다 | 같은 thread 로 `stream(null)` 을 부르면 멈춘 자리부터 돈다 |
| ④ 승인 | 셋을 묶는다. y 면 이어 가고, 아니면 멈춘 채로 둔다 |

그래프는 앞 시간에 배운 것을 활용합니다

In [8]:
import "npm:@langchain/core@1.2.14";
import { Annotation, StateGraph, START, END, MemorySaver } from "npm:@langchain/langgraph@1.4.18";

// 접속 정보: 실습과 같은 lab/.env 를 읽는다 (OpenRouter · qwen/qwen3-32b). 키는 화면에 찍지 않는다
const ENV: Record<string, string> = {};
try {
  for (const line of Deno.readTextFileSync("../lab/.env").split("\n")) {
    const m = line.match(/^\s*([A-Z_]+)\s*=\s*(.*)\s*$/);
    if (m) ENV[m[1]] = m[2];
  }
} catch { /* lab/.env 가 없으면 아래 기본값 */ }
const BASE_URL = ENV.LLM_BASE_URL ?? "https://openrouter.ai/api/v1";
const API_KEY  = ENV.LLM_API_KEY  ?? "";
const MODEL    = ENV.LLM_MODEL    ?? "qwen/qwen3-32b";
const OPTIONS  = { temperature: 0 };   // 시연이 매번 같게
console.log({ BASE_URL, MODEL, 키: API_KEY ? "있음" : "없음 — lab/.env 에 LLM_API_KEY 를 넣는다" });

// LangSmith (강사 시연용): demo/.env 에 키가 있으면 실행 기록(trace)이 LangSmith 에 남는다. 실습은 lab/.env 를 써서 남지 않는다
import { traceable } from "npm:langsmith@0.11.1/traceable";
try {
  for (const line of Deno.readTextFileSync(".env").split("\n")) {
    const m = line.match(/^\s*(LANGSMITH_[A-Z_]+)\s*=\s*(?:"([^"]*)"|([^#\s]*))/);   // "따옴표 값" 은 그대로, 아니면 # 주석 앞까지
    const value = m?.[2] ?? m?.[3];
    if (m && value) Deno.env.set(m[1], value);
  }
} catch { /* demo/.env 가 없으면 기록하지 않는다 */ }
if (!Deno.env.get("LANGSMITH_API_KEY")) Deno.env.set("LANGSMITH_TRACING", "false");   // 키가 없으면 끈다
// 모델을 부르는 함수를 감싸면 trace 에 "모델 호출" 한 칸으로 남는다. 기록을 안 켜면 그냥 원래 함수로 돈다
const asLLM = <F extends (...a: any[]) => Promise<any>>(fn: F, name: string) => traceable(fn, {
  name, run_type: "llm",
  processInputs: (i: any) => ({ messages: i.args ? i.args[0] : i.input }),
  processOutputs: (o: any) => ({ choices: [{ message: typeof o.outputs === "string" ? { role: "assistant", content: o.outputs } : o }] }),
});
console.log({ LangSmith: Deno.env.get("LANGSMITH_TRACING") === "true" && Deno.env.get("LANGSMITH_API_KEY") ? `기록함 · project ${Deno.env.get("LANGSMITH_PROJECT") ?? "default"}` : "기록 안 함 (demo/.env)" });

// 그래프 그림: 노드·엣지 이름을 mermaid.ink 로 보내 PNG 로 받아 셀 출력에 띄운다. 받지 못하면 mermaid 글자로 찍는다
async function showGraph(graph: any, options: { xray?: boolean } = {}) {
  const drawable = await graph.getGraphAsync(options);   // xray: true 면 subgraph 안의 노드까지 그린다
  try {
    const png = await drawable.drawMermaidPng();
    await Deno.jupyter.display(Deno.jupyter.image(new Uint8Array(await png.arrayBuffer())));
  } catch {
    console.log(drawable.drawMermaid());
  }
}

{
  BASE_URL: "https://openrouter.ai/api/v1",
  MODEL: "qwen/qwen3-32b",
  "키": "있음"
}
{ LangSmith: "기록함 · project AI Agent " }


## 준비: 오후 ① 의 튜터 Agent

모델 부르기(`chat()`), 사전 도구(`lookup_word`), 그래프는 오후 ① · ② 에서 쓴 것 그대로입니다. tutor 가 모델을 부르고, 도구를 원하면 tools 로 갔다가 다시 tutor 로 돌아옵니다. 이번에도 `compile()` 하기 전의 `builder` 까지만 만들어 둡니다. 같은 그래프를 여러 방식으로 compile 하려고요.

In [9]:
// 오전에 만든 chat() 그대로 — 다시 보낼 칸(role · content · tool_calls)만 돌려준다
async function callModel(messages: any[], tools?: any[]) {
  const res = await fetch(`${BASE_URL}/chat/completions`, {
    method: "POST",
    headers: { "Content-Type": "application/json", Authorization: `Bearer ${API_KEY}` },
    body: JSON.stringify({ model: MODEL, messages, tools, ...OPTIONS }),
  });
  const { role, content, tool_calls } = (await res.json()).choices[0].message;
  return { role, content: content?.trim() ?? null, tool_calls };
}
const chat = asLLM(callModel, "chat");   // LangSmith 에 모델 호출로 남긴다

// 도구 하나: 단어 뜻과 쉬운 말 찾기 (작은 사전이라 시연이 매번 같다)
const DICT: Record<string, { 뜻: string; 쉬운말: string }> = {
  ubiquitous: { 뜻: "어디에나 있는", 쉬운말: "everywhere" },
  mitigate:   { 뜻: "완화하다, 줄이다", 쉬운말: "reduce" },
  meticulous: { 뜻: "꼼꼼한", 쉬운말: "careful" },
  detrimental:{ 뜻: "해로운", 쉬운말: "harmful" },
};
const tools = [{
  type: "function",
  function: {
    name: "lookup_word",
    description: "영어 단어 하나의 뜻과, 바꿔 쓸 쉬운 말을 사전에서 찾는다",
    parameters: { type: "object", properties: { word: { type: "string", description: "찾을 영어 단어 하나" } }, required: ["word"] },
  },
}];
function lookupWord(word: string) {
  const w = word.toLowerCase().trim();
  return DICT[w] ? { word: w, ...DICT[w] } : { word: w, error: "사전에 없는 단어" };
}

const SYSTEM = { role: "system", content: "너는 IELTS 쓰기를 돕는 영어 튜터다. 단어 뜻은 추측하지 말고 lookup_word 로 확인한다. 한국어로 짧게 답한다." };
const ask = (q: string) => ({ messages: [SYSTEM, { role: "user", content: q }] });
// 메시지를 보기 좋게 찍는다 (tutor-stream 노트북의 pretty 그대로)
const TITLE: Record<string, string> = { system: "System Message", user: "Human Message", assistant: "Ai Message", tool: "Tool Message" };
function pretty(m: any) {
  const title = ` ${TITLE[m.role]} `;
  const left = Math.floor((60 - title.length) / 2);
  console.log("=".repeat(left) + title + "=".repeat(60 - title.length - left));
  if (m.tool_calls?.length) {
    const calls = m.tool_calls.map((c: any) => `${c.function.name}(${c.function.arguments})`);
    console.log("Tool Calls:", calls.join(", "));
  } else console.log(m.content);
}

In [10]:
// 오후 ① 의 튜터 Agent 그대로: tutor ⇢ tools → tutor
const State = Annotation.Root({
  messages: Annotation<any[]>({ reducer: (prev, next) => [...prev, ...next], default: () => [] }),
});
async function tutor(state: typeof State.State) {
  return { messages: [await chat(state.messages, tools)] };
}
function toolsNode(state: typeof State.State) {
  const calls = state.messages.at(-1).tool_calls;
  console.log(`▶ tools   ${calls.map((c: any) => c.function.arguments).join(" · ")}`);
  return { messages: calls.map((c: any) => ({
    role: "tool", tool_call_id: c.id, content: JSON.stringify(lookupWord(JSON.parse(c.function.arguments).word)),
  })) };
}
function route(state: typeof State.State) {
  return state.messages.at(-1).tool_calls?.length ? "tools" : END;
}
const builder = new StateGraph(State)
  .addNode("tutor", tutor)
  .addNode("tools", toolsNode)
  .addEdge(START, "tutor")
  .addConditionalEdges("tutor", route, ["tools", END])
  .addEdge("tools", "tutor");

## ① 멈춘다 — breakpoint

compile 할 때 `interruptBefore` 에 노드 이름을 주면, 그래프는 그 노드를 **돌기 직전에** 멈춥니다. 도구를 쓰기 전에 사람이 확인하고 싶으니 tools 앞에서 멈춰 보겠습니다. 노드를 돈 직후에 멈추는 `interruptAfter` 도 있습니다.

In [ ]:
// ① tools 앞에서 멈춘다: compile 에 interruptBefore. 멈췄다 이어 가려면 checkpointer 가 있어야 한다
const approveAgent = builder.compile({ checkpointer: new MemorySaver(), interruptBefore: ["tools"] });
await showGraph(approveAgent);
const thread1 = { configurable: { thread_id: "breakpoints-1" } };

// stream 으로 돌린다 (values: 각 단계마다 그때의 State 전체를 돌려준다.)
for await (const state of await approveAgent.stream(ask("ubiquitous 뜻을 사전에서 찾아 줘"), {
  ...thread1,
  streamMode: "values",
})) {
  if (state.__interrupt__) console.log("⏸ __interrupt__ 받았다. 그래프 실행이 여기서 끝난다");   // 멈추면 State 대신 이게 온다
  else {
    for (const m of state.messages) if (m.role !== "system") pretty(m);   // system 메시지는 매번 같으니 뺀다
    console.log("-".repeat(60));   // State 하나 끝
  }
}

**돌려 보면** State 가 두 번 옵니다. 처음 넣은 질문만 있는 State, 그리고 tutor 가 도구 요청을 붙인 State. 그 뒤에 State 대신 `__interrupt__` 가 오고 루프가 끝납니다. 사전은 아직 찾지 않았습니다.

## ② 멈춘 자리를 본다 — getState

그럼 어디서 멈춘 걸까요? `getState` 로 봅니다. `next` 가 중요합니다. 다음에 돌 노드입니다. 값이 있으면 그 노드 앞에서 멈춘 것이고, 빈 배열 `[]` 이면 끝까지 돈 것입니다.

In [18]:
// 멈춘 자리를 본다: getState 의 next 가 다음에 돌 노드다
const snapshot = await approveAgent.getState(thread1);
console.log("다음 노드:", snapshot.next);
console.log("모델이 하려는 일:", snapshot.values.messages.at(-1).tool_calls.map((c: any) => `${c.function.name} ${c.function.arguments}`));

다음 노드: [ "tools" ]
모델이 하려는 일: [ 'lookup_word {"word": "ubiquitous"}' ]


**돌려 보면** 다음 노드는 `tools`, 모델이 하려는 일은 `lookup_word {"word": "ubiquitous"}` 입니다. 사람은 이걸 보고 승인할지 정하면 됩니다.

## ③ 이어 간다 — null

이어 가려면 같은 thread 로, 입력 자리에 `null` 을 넣어 stream 을 다시 부릅니다. "새 입력은 없고, 멈춘 자리에서 계속" 이라는 뜻입니다. 앞으로 여러 번 부르니 ① 의 루프를 `run` 함수로 묶어 둡니다.

In [ ]:
// ① 의 stream 루프를 함수로 묶는다: State 를 받을 때마다 찍고, 멈추면 표시한다
async function run(graph: any, input: any, thread: any) {
  for await (const state of await graph.stream(input, { ...thread, streamMode: "values" })) {
    if (state.__interrupt__) console.log("⏸ __interrupt__ · 다음 노드:", (await graph.getState(thread)).next);
    else {
      for (const m of state.messages) if (m.role !== "system") pretty(m);   // system 메시지는 매번 같으니 뺀다
      console.log("-".repeat(60));   // State 하나 끝
    }
  }
}

// 이어 가기: 입력 자리에 null 을 넣고 같은 thread 로 부른다. 멈춘 자리(tools)부터 돈다
await run(approveAgent, null, thread1);

**돌려 보면** 첫 State 는 멈춘 자리의 State 입니다(values 는 지금 State 부터 줍니다). 그래서 질문과 도구 요청이 한 번 더 보입니다. 그다음 tools 가 사전 결과를 붙인 State, tutor 가 답을 붙인 State 가 옵니다. 이번에는 `__interrupt__` 없이 루프가 끝났습니다. 끝까지 돈 것입니다.

## ④ 승인 — 멈추기 · 보기 · 이어 가기

이 셋을 묶으면 승인 흐름이 됩니다. 사람이 y 면 이어 가고, 아니면 멈춘 채로 둡니다.

In [ ]:
// 승인 ①: 멈출 때까지 돌리고, 모델이 하려는 일을 사람에게 보여 준다
const thread2 = { configurable: { thread_id: "breakpoints-2" } };
await run(approveAgent, ask("mitigate 뜻을 사전에서 찾아 줘"), thread2);
const pending = (await approveAgent.getState(thread2)).values.messages.at(-1).tool_calls;
console.log("사전을 찾으려고 한다:", pending.map((c: any) => c.function.arguments).join(" · "));

사람에게 직접 묻습니다. `prompt()` 는 Python 의 `input()` 과 같습니다. 셀을 돌리면 노트북 위쪽에 입력창이 뜹니다. y 면 이어 가고, 아니면 멈춘 채로 둡니다.

In [ ]:
// 승인 ②: 사람에게 묻는다. y 면 이어 가고, 아니면 멈춘 채로 둔다
const approval = prompt("사전을 찾을까요? (y/n)")?.trim();   // 노트북 위쪽에 입력창이 뜬다
if (approval === "y") await run(approveAgent, null, thread2);
else console.log("승인하지 않았다. 다음 노드:", (await approveAgent.getState(thread2)).next);

승인 ① 부터 다시 돌리고, 이번에는 `n` 을 입력해 보세요. 그래프는 tools 앞에서 멈춘 채로 남아 있습니다. 거절이나 의견을 그래프에 알려 주려면 State 를 고쳐야 합니다. 다음 노트북 `tutor-edit-state.ipynb` 에서 봅니다.